# 05 - ONNX export and verification (praxis Step 7)

Exports the 7 compressed/baseline MA1DCNN variants to ONNX on Colab and checks each file against its PyTorch model on the **real validation windows**. Only the finished `.onnx` files go to the Jetson.

| Variant | Source checkpoint (Drive `data/`) | Check used |
|---|---|---|
| baseline | `ma1dcnn_trial13.pt` | max logit difference < 1e-4 |
| recovered 20/40/60 | `ma1dcnn_trial13_prunedXX_recovered.pt` | max logit difference < 1e-4 |
| QAT 20/40/60 | `ma1dcnn_trial13_prunedXX_quantized_qat.pt` | same macro-F1, under 0.1% decision flips, mean logit difference under 0.01 |

QAT files are accepted on decisions, not on the maximum logit difference: INT8 rounding differs on a few values, so a max-difference limit fails even though the F1 is identical. The QAT limits were set after the first real run (66-81 flips of 875,885, F1 equal to four decimals), not before it.

**Run on:** Colab, CPU is enough. **Needs:** Google Drive with the project folder (checkpoints in `data/`, windowed validation data in `data/windowed/`) and the repo cloned from GitHub (the cell below does it); `export_onnx.py` is imported from the clone as `edge_fault_dx.deployment.export_onnx`.
**Writes:** `data/onnx/*.onnx` in the Drive project folder. Checkpoints are only read.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q onnx onnxruntime
!test -d /content/repo || git clone https://github.com/iodawa/edge-ai-fault-diagnosis-aerospace.git /content/repo

Mounted at /content/drive
Cloning into '/content/repo'...
remote: Enumerating objects: 374, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 374 (delta 22), reused 0 (delta 0), pack-reused 319 (from 1)
Receiving objects: 100% (374/374), 4.22 MiB | 21.20 MiB/s, done.
Resolving deltas: 100% (161/161), done.


In [2]:
import sys, os, glob, hashlib
PROJECT = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace"
sys.path.insert(0, "/content/repo/src")   # the GitHub clone; nothing from Drive goes on sys.path

import torch, onnx, onnxruntime as ort
from edge_fault_dx.deployment import export_onnx as ex
from edge_fault_dx.training.train import (build_dataloaders, evaluate,
                                          compute_metrics, TRIAL_13_TRAIN_CONFIG)

print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| onnx", onnx.__version__, "| onnxruntime", ort.__version__)

D = PROJECT + "/data/"
OUT = D + "onnx/"; os.makedirs(OUT, exist_ok=True)
_, val_loader = build_dataloaders(windowed_dir=D + "windowed")   # 175,177 validation windows

python 3.13.18 | torch 2.14.1+cpu | onnx 1.23.2 | onnxruntime 1.32.0


## 1. What the checkpoint files are
The two baseline files and the recovered file are weight files; the QAT file is a whole pickled model. `export_onnx.load_model` handles each kind.

In [3]:
!wc -c /content/repo/src/edge_fault_dx/deployment/export_onnx.py
!git -C /content/repo log -3 --oneline

12909 /content/repo/src/edge_fault_dx/deployment/export_onnx.py
9e2802c (HEAD -> main, origin/main, origin/HEAD) Add files via upload
f2a4051 Add files via upload
5341b07 Add files via upload


In [4]:
a = torch.load(D + "ma1dcnn_trial13.pt", map_location="cpu")
b = torch.load(D + "ma1dcnn_trial13_prod.pt", map_location="cpu")
print("baseline files identical:",
      list(a.keys()) == list(b.keys()) and all(torch.equal(a[k], b[k]) for k in a))

r = torch.load(D + "ma1dcnn_trial13_pruned20_recovered.pt", map_location="cpu", weights_only=False)
print("recovered20 pruned_config:", r["pruned_config"], " ratio:", r["ratio"])

q = ex.load_model(D + "ma1dcnn_trial13_pruned20_quantized_qat.pt", "qat")
print("qat20 ->", type(q).__name__)

baseline files identical: False
recovered20 pruned_config: {'n_input_channels': 28, 'channel_1': 26, 'channel_2': 51, 'channel_3': 102, 'kernel_size': 5, 'fc_units': 16, 'dropout': 0.5945728900118123, 'n_outputs': 5, 'conv_dropout': 0.33869914326562556, 'use_batchnorm': False}  ratio: 0.2
qat20 -> QuantizedConvTrunk


/usr/local/lib/python3.13/dist-packages/torch/_utils.py:505: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,
/usr/local/lib/python3.13/dist-packages/torch/_utils.py:499: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:111.)
  tensor = torch._empty_per_channel_affine_quantized(


## 2. Export all 7 variants and verify each
Each file is exported (opset 17, `dynamo=False`), checked against PyTorch with the check for its kind, and its operator list is printed. The operator list is the `check_ops` result: no `Attention` / `MultiHeadAttention` operators and no `com.microsoft` domain means the ONNX Runtime attention build flags can be turned off.

In [5]:
variants = [("baseline", "ma1dcnn_trial13.pt", "baseline", None)]
for r in (20, 40, 60):
    variants.append((f"recovered{r}", f"ma1dcnn_trial13_pruned{r}_recovered.pt", "recovered", None))
for r in (20, 40, 60):
    variants.append((f"qat{r}", f"ma1dcnn_trial13_pruned{r}_quantized_qat.pt", "qat", ex.QAT_THRESHOLDS[r]))

for name, fname, kind, thr in variants:
    print("=====", name)
    model = ex.load_model(D + fname, kind)
    ex.export_onnx(model, OUT + f"{name}.onnx")
    ex.verify(model, OUT + f"{name}.onnx", kind, val_loader, thr)
    ex.list_ops(OUT + f"{name}.onnx")

===== baseline
parity OK -- max abs diff 6.91e-06 < 0.0001 over 20 batches
op counts: {'Conv': 4, 'Relu': 5, 'Shape': 3, 'Constant': 10, 'Gather': 3, 'GlobalAveragePool': 2, 'Unsqueeze': 5, 'Concat': 3, 'Reshape': 3, 'Gemm': 4, 'Sigmoid': 2, 'Mul': 2}
opset domains: ['ai.onnx']  node domains: ['ai.onnx']
no Attention/MultiHeadAttention ops and no com.microsoft domain in this graph.
===== recovered20
parity OK -- max abs diff 5.72e-06 < 0.0001 over 20 batches
op counts: {'Conv': 4, 'Relu': 5, 'Shape': 3, 'Constant': 10, 'Gather': 3, 'GlobalAveragePool': 2, 'Unsqueeze': 5, 'Concat': 3, 'Reshape': 3, 'Gemm': 4, 'Sigmoid': 2, 'Mul': 2}
opset domains: ['ai.onnx']  node domains: ['ai.onnx']
no Attention/MultiHeadAttention ops and no com.microsoft domain in this graph.
===== recovered40
parity OK -- max abs diff 7.63e-06 < 0.0001 over 20 batches
op counts: {'Conv': 4, 'Relu': 5, 'Shape': 3, 'Constant': 10, 'Gather': 3, 'GlobalAveragePool': 2, 'Unsqueeze': 5, 'Concat': 3, 'Reshape': 3, 'Gemm':

/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


  threshold 0.425: max abs diff 0.6051, mean 0.002112, 99.9th pct 0.0619
  decision flips: 82 of 875885 (0.0094%)
  macro-F1 PyTorch 0.8385  ONNX 0.8384
decision parity OK
op counts: {'Constant': 49, 'QuantizeLinear': 7, 'Cast': 10, 'DequantizeLinear': 13, 'ConstantOfShape': 3, 'Conv': 4, 'Relu': 5, 'Shape': 2, 'Gather': 2, 'GlobalAveragePool': 2, 'Unsqueeze': 5, 'Concat': 3, 'Reshape': 3, 'Gemm': 4, 'Sigmoid': 2, 'Mul': 2}
opset domains: ['ai.onnx']  node domains: ['ai.onnx']
no Attention/MultiHeadAttention ops and no com.microsoft domain in this graph.
===== qat40


/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


  threshold 0.425: max abs diff 0.4789, mean 0.001808, 99.9th pct 0.0581
  decision flips: 66 of 875885 (0.0075%)
  macro-F1 PyTorch 0.8306  ONNX 0.8306
decision parity OK
op counts: {'Constant': 49, 'QuantizeLinear': 7, 'Cast': 10, 'DequantizeLinear': 13, 'ConstantOfShape': 3, 'Conv': 4, 'Relu': 5, 'Shape': 2, 'Gather': 2, 'GlobalAveragePool': 2, 'Unsqueeze': 5, 'Concat': 3, 'Reshape': 3, 'Gemm': 4, 'Sigmoid': 2, 'Mul': 2}
opset domains: ['ai.onnx']  node domains: ['ai.onnx']
no Attention/MultiHeadAttention ops and no com.microsoft domain in this graph.
===== qat60


/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


  threshold 0.4: max abs diff 0.4628, mean 0.002028, 99.9th pct 0.0671
  decision flips: 78 of 875885 (0.0089%)
  macro-F1 PyTorch 0.8202  ONNX 0.8202
decision parity OK
op counts: {'Constant': 49, 'QuantizeLinear': 7, 'Cast': 10, 'DequantizeLinear': 13, 'ConstantOfShape': 3, 'Conv': 4, 'Relu': 5, 'Shape': 2, 'Gather': 2, 'GlobalAveragePool': 2, 'Unsqueeze': 5, 'Concat': 3, 'Reshape': 3, 'Gemm': 4, 'Sigmoid': 2, 'Mul': 2}
opset domains: ['ai.onnx']  node domains: ['ai.onnx']
no Attention/MultiHeadAttention ops and no com.microsoft domain in this graph.


## 3. Which baseline file?
`ma1dcnn_trial13.pt` is the checkpoint behind the Phase 4 baseline (0.8440). `ma1dcnn_trial13_prod.pt` is a separate training run of the same configuration (standalone `train.py` run). The check below confirms which file scores 0.8440 and that the exported baseline reproduces it.

In [6]:
thr = TRIAL_13_TRAIN_CONFIG["threshold"]
loss_fn = torch.nn.BCEWithLogitsLoss()
for fname in ["ma1dcnn_trial13.pt", "ma1dcnn_trial13_prod.pt"]:
    m = ex.load_model(D + fname, "baseline")
    r = evaluate(m, val_loader, loss_fn, torch.device("cpu"), thr)
    print(fname, "macro-F1:", round(r["f1_macro"], 4))

sess = ort.InferenceSession(OUT + "baseline.onnx", providers=["CPUExecutionProvider"])
ys, ls = [], []
for x, y in val_loader:
    ls.append(torch.from_numpy(sess.run(["logits"], {"window": x.numpy()})[0])); ys.append(y)
print("ONNX baseline macro-F1:", round(compute_metrics(torch.cat(ys), torch.cat(ls), thr)["f1_macro"], 4))

ma1dcnn_trial13.pt macro-F1: 0.844
ma1dcnn_trial13_prod.pt macro-F1: 0.8401
ONNX baseline macro-F1: 0.844


## 4. Fingerprints
Record these with the results. After copying the `.onnx` files to the Jetson, `sha256sum` there should start with the same 16 characters.

In [7]:
def sha(p): return hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
for name, fname, kind, thr in variants:
    print("checkpoint", fname, sha(D + fname))
for f in sorted(glob.glob(OUT + "*.onnx")):
    print("onnx", os.path.basename(f), os.path.getsize(f), "bytes", sha(f))

checkpoint ma1dcnn_trial13.pt b8e027d44f244867
checkpoint ma1dcnn_trial13_pruned20_recovered.pt 68f515f9c2f51ecd
checkpoint ma1dcnn_trial13_pruned40_recovered.pt 9828e7ba7cfb07ac
checkpoint ma1dcnn_trial13_pruned60_recovered.pt 3da6b009f5bca187
checkpoint ma1dcnn_trial13_pruned20_quantized_qat.pt a309609780925f15
checkpoint ma1dcnn_trial13_pruned40_quantized_qat.pt 1bae1d62b5bf801a
checkpoint ma1dcnn_trial13_pruned60_quantized_qat.pt 176f2b5a802cd475
onnx baseline.onnx 271763 bytes d03b99cc1ca4e0b9
onnx qat20.onnx 81236 bytes 465f22ef2d2d3f04
onnx qat40.onnx 54746 bytes 0cabfbcea72457cb
onnx qat60.onnx 34697 bytes ddbd966f4f73ef05
onnx recovered20.onnx 179698 bytes d2ce1f9fe948d15c
onnx recovered40.onnx 107476 bytes e23646a0c5c44d48
onnx recovered60.onnx 55550 bytes 844b5b68a81c7bdf
